In [1]:
#RGAPI-060e4fb1-4dfe-4abe-b764-12b19e269ec7

In [2]:
import requests
import time
from tqdm import tqdm
import json

# Riot API 설정
API_KEY = 'RGAPI-060e4fb1-4dfe-4abe-b764-12b19e269ec7'  # 🔴 ← 본인의 API 키로 바꾸기
REGION = 'asia'  # account-v1, match-v5 용
PLATFORM = 'kr'  # summoner-v4 용 (예전 방식)
HEADERS = {'X-Riot-Token': API_KEY}


In [3]:
def get_puuid_by_riot_id(game_name, tag_line):
    """
    최신 Riot ID (닉네임#태그) 기반으로 PUUID 가져오기
    """
    url = f'https://{REGION}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}'
    res = requests.get(url, headers=HEADERS)
    if res.status_code == 200:
        return res.json()['puuid']
    else:
        print(f"[!] Failed to get puuid for {game_name}#{tag_line}: {res.status_code}")
        return None


In [4]:
def get_match_ids(puuid, count=20):
    url = f'https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start=0&count={count}'
    res = requests.get(url, headers=HEADERS)
    if res.status_code == 200:
        return res.json()
    else:
        return []

def get_match_data(match_id):
    url = f'https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}'
    res = requests.get(url, headers=HEADERS)
    if res.status_code == 200:
        return res.json()
    else:
        return None


In [5]:


def collect_random_matches(target_count=100, seed_ids=None):
    seen_matches = set()
    seen_puuids = set()
    queue_puuids = set()

    print("[1] 초기 Riot ID seed로 시작합니다...")
    for game_name, tag_line in seed_ids:
        puuid = get_puuid_by_riot_id(game_name, tag_line)
        if puuid:
            queue_puuids.add(puuid)
        time.sleep(1.2)

    all_matches = []

    print("[2] 무작위 매치 수집 중...")
    with tqdm(total=target_count) as pbar:
        while len(all_matches) < target_count and queue_puuids:
            current_puuid = queue_puuids.pop()
            if current_puuid in seen_puuids:
                continue
            seen_puuids.add(current_puuid)

            match_ids = get_match_ids(current_puuid)
            time.sleep(1.2)

            for match_id in match_ids:
                if match_id in seen_matches:
                    continue

                match_data = get_match_data(match_id)
                time.sleep(1.2)

                if match_data:
                    all_matches.append(match_data)
                    seen_matches.add(match_id)
                    pbar.update(1)

                    participants = match_data.get('metadata', {}).get('participants', [])
                    for p in participants:
                        if p not in seen_puuids:
                            queue_puuids.add(p)

                if len(all_matches) >= target_count:
                    break

    print(f"\n✅ 총 {len(all_matches)}개의 매치 데이터를 수집했습니다.")
    return all_matches


In [6]:
# 예시 Riot ID 목록 (닉네임#태그)
RIOT_IDS = [
    ('즐겁게게임하는놈', 'KR1'),
    ('프로딸잡이', 'KR1')
]

# 100개의 매치 데이터를 수집
matches = collect_random_matches(target_count=10, seed_ids=RIOT_IDS)


[1] 초기 Riot ID seed로 시작합니다...
[2] 무작위 매치 수집 중...


100%|██████████| 10/10 [00:15<00:00,  1.50s/it]


✅ 총 10개의 매치 데이터를 수집했습니다.


In [13]:
with open('random_matches.jsonl', 'w', encoding='utf-8') as f:
    for match in matches:
        json.dump(match, f, ensure_ascii=False)
        f.write('\n')

print("✅ 저장 완료: random_matches.jsonl")


✅ 저장 완료: random_matches.jsonl


In [11]:
matches

[{'metadata': {'dataVersion': '2',
   'matchId': 'KR_7611002827',
   'participants': ['_RB9bBSOUco2StOlUACgu7ohzg3PPgWZbMKW2FyuRf3-9Khr0I4Tj2bZMe9Qnh0W-t69K5-1u7TQCA',
    'qv3f-2gfYnkoS_sYg88qXZEElHMhGVYL1SGsRzy-KQASXybgNk3e6B3zzJbu1PsfS-ha-mTr4EGrOw',
    'aUT-DbH3fu95--N_3MoW-Kko1peC8YWkQbtJ1IvWm0QpsTYnrYYhtvnKLmkwaU4tcFP4QY_8EzyDuQ',
    'Qy-PKU7_5OfdT5PWznONXLR1Sr1On-pgkuDa9ShZzmuBArt0A2_CSpVsU-CDwgbblxAU2VQfwJ-NZQ',
    'rHvA63Dr4XO19Zd-8TlyOfFuZtZ9N_wQHhC-FGuJTfZHqR9CRsWqyiDvk-tc5bcQkqSiaveGcOl1jQ',
    'ENiNke6CeefjknaOi7JY9PXy9PVr4XY6UmQc63FACZWBeP0jn_Hwn5tYtMrAry09QhBIOTXES1ASHw',
    '3ZH0UmgiIDIfTlI7ZVATIemy9QCSszfuFiqWMGWX-KqAtiJagsZc760eaUmvY8DPyXRHTAp4pAEo8g',
    'RHgDXJo8o2P7-ob3Nhg8M3Qu9rCc7gTycHyUXa_qWKO_34-AOrHUO2FM-nRJnxlFwqWcOWQht0bT1w',
    '7ZDw2Z1liqe8FNmbTqTj7MBgA6WaSM92A0jW3RGiG5GcPs-nmOw1Gm_TQ2moz9l64NtfzH7fc_JLXg',
    'yYjh7i6BeQSrTclgxymxJfV_9-ATeEDfEGhlBTqnxq5U19mMfOmkBS4UBd0mxDRU8jPX9GCDLbWpow']},
  'info': {'endOfGameResult': 'GameComplete',
   'gameCr

In [17]:
import json
print(json.dumps(matches[0]['info']['participants'][0], indent=2))


{
  "PlayerScore0": 0,
  "PlayerScore1": 0,
  "PlayerScore10": 0,
  "PlayerScore11": 0,
  "PlayerScore2": 0,
  "PlayerScore3": 0,
  "PlayerScore4": 0,
  "PlayerScore5": 0,
  "PlayerScore6": 0,
  "PlayerScore7": 0,
  "PlayerScore8": 0,
  "PlayerScore9": 0,
  "allInPings": 0,
  "assistMePings": 1,
  "assists": 10,
  "baronKills": 0,
  "basicPings": 0,
  "bountyLevel": 6,
  "challenges": {
    "12AssistStreakCount": 0,
    "HealFromMapSources": 0,
    "InfernalScalePickup": 0,
    "SWARM_DefeatAatrox": 0,
    "SWARM_DefeatBriar": 0,
    "SWARM_DefeatMiniBosses": 0,
    "SWARM_EvolveWeapon": 0,
    "SWARM_Have3Passives": 0,
    "SWARM_KillEnemy": 0,
    "SWARM_PickupGold": 0,
    "SWARM_ReachLevel50": 0,
    "SWARM_Survive15Min": 0,
    "SWARM_WinWith5EvolvedWeapons": 0,
    "abilityUses": 301,
    "acesBefore15Minutes": 0,
    "alliedJungleMonsterKills": 8,
    "baronTakedowns": 0,
    "blastConeOppositeOpponentCount": 0,
    "bountyGold": 0,
    "buffsStolen": 1,
    "completeSupportQues